# Targeted ARBITER fine-tuning

Prepare and audit data, adapt the model, then compare against the frozen base.
See [the full plan](../docs/targeted_finetuning.md). This notebook does not contain an
80k training dataset or claim improved scores. Run from a checkout installed with
`pip install -e ".[finetuning,benchmark]"`. No GPU training starts by default.


In [ ]:
from pathlib import Path
import json, subprocess, sys
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "finetuning/train.py").exists(), "Open this notebook inside the ARBITER checkout."
BASE = ROOT / "checkpoints/arbiter-272m"
TRAIN = ROOT / "results/adaptation/train.jsonl"
VALIDATION = ROOT / "results/adaptation/validation.jsonl"
OUTPUT = ROOT / "results/arbiter-targeted-v1"
HELDOUT = [ROOT / "benchmark/data/cases.jsonl", ROOT / "benchmark/data/public_requested_v1/cases.jsonl"]
RUN_PREFLIGHT = False
RUN_TRAINING = False
RUN_COMPARISON = False
RUN_BENCHMARK = False


## 1. Prepare the data

Target: 16k Typed/workflow + 24k NLI + 24k new policy + 16k replay decisions.
These are target budgets, not actual downloaded counts. Typed's official train split
alone is too small; new reviewed scenarios are needed. Keep translations/questions/
variants of a scenario in the same split. Public Fast Decisions remains evaluation-only.

The source converter supports pinned Typed train and XNLI translated train. Curate its
outputs and your policy/replay data before using the training command. See the plan for
source commands, license review and JSONL schema. No automatic synthetic labeling.


In [ ]:
from collections import Counter
if TRAIN.exists() and VALIDATION.exists():
    from finetuning.data import read_rows, audit
    train_rows, val_rows = read_rows(TRAIN), read_rows(VALIDATION)
    print(json.dumps(audit(train_rows, val_rows, HELDOUT), indent=2))
    print("Training groups:", len({r["group_id"] for r in train_rows}))
    print("Validation families:", dict(Counter(r["family"] for r in val_rows)))
else:
    print("Create the curated train.jsonl and validation.jsonl first; see the plan.")


## 2. Preflight and train

Architecture and loss unchanged. FP32 master weights, CUDA BF16 autocast; effective
batch 32, one epoch, clipping 3. No silent truncation. Best validation package and latest
resumable optimizer checkpoint only. Public benchmark data never select checkpoints.


In [ ]:
common = [sys.executable, "-m", "finetuning.train", "--model", str(BASE),
          "--train", str(TRAIN), "--validation", str(VALIDATION),
          "--heldout", *map(str, HELDOUT)]
preflight = common + ["--output", str(ROOT / "results/adaptation/preflight"), "--validate-only"]
command = common + ["--output", str(OUTPUT), "--device", "cuda", "--epochs", "1",
                    "--batch-size", "4", "--accumulation", "8", "--max-length", "1024",
                    "--backbone-lr", "5e-6", "--head-lr", "2e-5", "--clip", "3", "--eval-every", "250"]
import shlex
print(shlex.join(command))
if RUN_PREFLIGHT:
    subprocess.run(preflight, cwd=ROOT, check=True)
if RUN_TRAINING:
    subprocess.run(command, cwd=ROOT, check=True)
# To resume a trusted local run, use exactly command + ["--resume"].


In [ ]:
print(f"tensorboard --logdir {OUTPUT / 'tensorboard'} --host 0.0.0.0 --port 6006")
metrics = OUTPUT / "validation.jsonl"
if metrics.exists():
    points = [json.loads(line) for line in metrics.read_text().splitlines()]
    print("Last validation:", json.dumps(points[-1], indent=2))
    print("Lowest grouped validation CE:", min(p["macro_ce"] for p in points))


## 3. Inspect both models, including expected answers

These hand-authored examples are demonstrations, not benchmark evidence. New unseen
policies and properly reserved examples are required for a transfer claim. Expected
answers are displayed beside each prediction, including errors.


In [ ]:
examples = [
    {"context": "Policy: escalate unresolved access failures. The account was locked yesterday, but access has been restored.",
     "instruction": "Should this access problem be escalated under the policy?",
     "question_type": "noul", "criteria": {"false": "Do not escalate.", "true": "Escalate."},
     "expected": "false"},
    {"context": "All approved invoices have been paid. Invoice A is approved.",
     "instruction": "Classify the hypothesis: Invoice A has been paid.", "question_type": "choice",
     "criteria": {"entailment": "Follows from the premise.", "neutral": "Cannot be determined.", "contradiction": "Contradicts the premise."},
     "expected": "entailment"},
    {"context": "Rubric: low = no affected users; medium = 1 to 9; high = 10 or more. Four users cannot log in.",
     "instruction": "Choose the severity according to the rubric.", "question_type": "score",
     "criteria": {"low": "No users.", "medium": "1 to 9 users.", "high": "10 or more users."},
     "expected": "medium"},
]
if RUN_COMPARISON:
    import gc
    from arbiter import ArbiterModel
    for name, package in [("base", BASE), ("adapted", OUTPUT / "best")]:
        model = ArbiterModel.from_pretrained(package, device="cpu")
        requests = [{k:v for k,v in e.items() if k != "expected"} for e in examples]
        for example, result in zip(examples, model.decide_batch(requests)):
            print(name, "expected:", example["expected"], "prediction:", result["selected_key"],
                  "correct:", result["selected_key"] == example["expected"], result["probabilities"])
        del model
        gc.collect()


## 4. Frozen before/after evaluation

Run only after selecting a candidate on validation. Same data and adapters for both
packages. Keep routing-retention and failed domains in the report. Scenario-grouped
paired confidence intervals and Multi-label set metrics require the existing report
analysis; this notebook does not fabricate those from aggregate accuracies.


In [ ]:
for name, package in [("base", BASE), ("adapted", OUTPUT / "best")]:
    for suite, data in [("historical", HELDOUT[0]), ("public", HELDOUT[1])]:
        cmd = [sys.executable, "-m", "benchmark.run", "--model", "arbiter",
               "--checkpoint", str(package), "--data", str(data), "--device", "cuda",
               "--output", str(ROOT / f"results/targeted-{name}-{suite}")]
        print(shlex.join(cmd))
        if RUN_BENCHMARK:
            subprocess.run(cmd, cwd=ROOT, check=True)
